## Note on data inputs

This notebook documents the **analysis method**. The raw licensed-facility inputs it reads (California Community Care Licensing / CDPH files containing facility administrator names, telephone numbers, and complaint histories) are **not distributed** with this repository, to protect vulnerable populations. De-identified and aggregated outputs are included; see `docs/DATA_AVAILABILITY.md`.

_As a result, this notebook is provided for transparency of method and will not run end-to-end against this public repository alone._

In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd

ROOT = Path("..")

DATA = ROOT / "data"

RAW = DATA / "raw"
PROCESSED = DATA / "processed"

In [ ]:
main_cols = [
    "Facility Type",
    "Facility Number",
    "Facility Name",
    "Licensee",
    "Facility Administrator",
    "Facility Telephone Number",
    "Facility Address",
    "Facility City",
    "Facility State",
    "Facility Zip",
    "County Name",
    "Regional Office",
    "Facility Capacity",
    "Facility Status",
    "License First Date",
    "Closed Date",
    "Last Visit Date",
    "Inspection Visits",
    "Complaint Visits",
    "Other Visits",
    "Total Visits",
    "Citation Numbers",
    "POC Dates",
    "All Visit Dates",
    "Inspection Visit Dates",
    "Inspect TypeA",
    "Inspect TypeB",
    "Other Visit Dates",
    "Other TypeA",
    "Other TypeB",
    "Complaint Type A",
    "Complaint Type B",
    "Total Allegations",
    "Inconclusive Allegations",
    "Substantiated Allegations",
    "Unsubstantiated Allegations",
    "Unfounded Allegations",
    "Complaint Info"
]

In [ ]:
child_res = pd.read_csv(
    RAW / "socialservices" / "24HourResidentialCareforChildren05172026.csv",
    usecols=range(38),
    names=main_cols,
    header=0
)

In [ ]:
child_res.head()

In [ ]:
child_res["Facility Type"].value_counts()

In [ ]:
child_res_eaton = child_res[
    child_res["Facility City"]
    .astype(str)
    .str.contains(
        "SIERRA|PASADENA|ALTADENA|CANADA",
        case=False,
        na=False
    )
].copy()

In [ ]:
len(child_res_eaton)

In [ ]:
child_res_eaton[
    [
        "Facility Name",
        "Facility Type",
        "Facility Capacity",
        "Facility Status",
        "Facility Address"
    ]
]

In [ ]:
child_res.columns.tolist()

In [ ]:
child_res.head()

In [ ]:
child_res["Facility City"].value_counts().head(30)

In [ ]:
childcare = pd.read_csv(
    RAW / "socialservices" / "ChildCareCenters05172026.csv",
    usecols=range(38),
    names=main_cols,
    header=0
)

In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd

ROOT = Path("..")

DATA = ROOT / "data"

RAW = DATA / "raw"
PROCESSED = DATA / "processed"


In [ ]:
childcare_eaton = childcare[
    childcare["Facility City"]
    .astype(str)
    .str.contains(
        "SIERRA|PASADENA|ALTADENA|CANADA",
        case=False,
        na=False
    )
].copy()

In [ ]:
with open(
    RAW / "socialservices" / "ChildCareCenters05172026.csv",
    "r",
    encoding="utf-8",
    errors="replace"
) as f:
    for i in range(5):
        print(repr(f.readline()))

In [ ]:
childcare = pd.read_csv(
    RAW / "socialservices" / "ChildCareCenters05172026.csv",
    engine="python",
    on_bad_lines="skip"
)

In [ ]:
childcare.columns.tolist()

In [ ]:
childcare.shape

In [ ]:
childcare.columns.tolist()


In [ ]:
childcare["Facility City"].value_counts().head(50)

In [ ]:
childcare_eaton = childcare[
    childcare["Facility City"]
    .astype(str)
    .str.contains(
        "SIERRA|PASADENA|ALTADENA|CANADA",
        case=False,
        na=False
    )
].copy()

len(childcare_eaton)


In [ ]:
childcare_eaton[
    [
        "Facility Name",
        "Facility City",
        "Facility Capacity",
        "Facility Status",
        "Closed Date",
        "Facility Address"
    ]
]

In [ ]:
childcare_eaton["full_address"] = (
    childcare_eaton["Facility Address"].astype(str)
    + ", "
    + childcare_eaton["Facility City"].astype(str)
    + ", CA "
    + childcare_eaton["Facility Zip"].astype(str)
)

In [ ]:
childcare_eaton[
    [
        "Facility Number",
        "Facility Address",
        "Facility City",
        "Facility State",
        "Facility Zip"
    ]
].to_csv(
    PROCESSED / "socialservices" / "childcare_eaton_census_input.csv",
    index=False,
    header=False
)

In [ ]:
childcare_geo = pd.read_csv(
    PROCESSED / "socialservices" / "childcare_eaton_geocoded.csv",
    header=None
)

childcare_geo.head()

In [ ]:
childcare_geo.columns = [
    "Facility Number",
    "input_address",
    "match_status",
    "match_type",
    "matched_address",
    "coordinates",
    "tiger_line_id",
    "side",
    "state_fips",
    "county_fips",
    "tract",
    "block"
]

In [ ]:
childcare_geo[["longitude", "latitude"]] = (
    childcare_geo["coordinates"]
    .str.split(",", expand=True)
    .astype(float)
)

In [ ]:
childcare_geocoded = childcare_eaton.merge(
    childcare_geo[
        [
            "Facility Number",
            "match_status",
            "match_type",
            "matched_address",
            "longitude",
            "latitude"
        ]
    ],
    on="Facility Number",
    how="left"
)

In [ ]:
childcare_gdf = gpd.GeoDataFrame(
    childcare_geocoded,
    geometry=gpd.points_from_xy(
        childcare_geocoded["longitude"],
        childcare_geocoded["latitude"]
    ),
    crs="EPSG:4326"
)

In [ ]:
perimeter = gpd.read_file(
    RAW / "perimeters" / "eaton_perimeter.geojson"
)

In [ ]:
childcare_gdf = childcare_gdf.to_crs(perimeter.crs)

childcare_in_fire = gpd.sjoin(
    childcare_gdf,
    perimeter,
    predicate="intersects"
)

len(childcare_in_fire)

In [ ]:
impacted = pd.read_csv(
    PROCESSED / "calfire" / "eaton_impacted_structures.csv"
)

In [ ]:
impacted["address_clean"] = (
    impacted["Site Address (parcel)"]
    .astype(str)
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.replace(" STREET", " ST", regex=False)
    .str.replace(" AVENUE", " AVE", regex=False)
    .str.replace(" DRIVE", " DR", regex=False)
    .str.replace(" ROAD", " RD", regex=False)
    .str.strip()
)

impacted["street_only"] = (
    impacted["address_clean"]
    .str.split(",")
    .str[0]
    .str.strip()
)

In [ ]:
childcare_in_fire["address_clean"] = (
    childcare_in_fire["Facility Address"]
    .astype(str)
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.replace(" STREET", " ST", regex=False)
    .str.replace(" AVENUE", " AVE", regex=False)
    .str.replace(" DRIVE", " DR", regex=False)
    .str.replace(" ROAD", " RD", regex=False)
    .str.strip()
)

In [ ]:
childcare_exact = childcare_in_fire.merge(
    impacted,
    left_on="address_clean",
    right_on="street_only",
    how="inner",
    suffixes=("_facility", "_calfire")
)

In [ ]:
childcare_exact[
    [
        "Facility Name",
        "Facility Address",
        "Site Address (parcel)",
        "* Damage"
    ]
].drop_duplicates()

In [ ]:
childcare_major = childcare_exact[
    childcare_exact["* Damage"].isin([
        "Major (25-50%)",
        "Destroyed (>50%)"
    ])
]

In [ ]:
childcare_major["Facility Name"].nunique()

In [ ]:
childcare_destroyed = childcare_exact[
    childcare_exact["* Damage"]
    == "Destroyed (>50%)"
]

In [ ]:
childcare_destroyed["Facility Name"].nunique()

In [ ]:
childcare_major[
    [
        "Facility Name",
        "Facility Capacity"
    ]
].drop_duplicates()["Facility Capacity"].sum()

In [ ]:
childcare_destroyed[
    [
        "Facility Name",
        "Facility Capacity"
    ]
].drop_duplicates()["Facility Capacity"].sum()

In [ ]:
childcare_export = (
    childcare_destroyed[
        [
            "Facility Name",
            "Facility Number",
            "Facility Capacity",
            "Facility Address",
            "Facility City",
            "Facility Status",
            "Closed Date",
            "latitude",
            "longitude",
            "* Damage"
        ]
    ]
    .drop_duplicates(subset=["Facility Name"])
)

In [ ]:
childcare_export.to_csv(
    PROCESSED / "socialservices" / "childcare_impacted_facilities.csv",
    index=False
)

In [ ]:
childcare_export_gdf = gpd.GeoDataFrame(
    childcare_export,
    geometry=gpd.points_from_xy(
        childcare_export["longitude"],
        childcare_export["latitude"]
    ),
    crs="EPSG:4326"
)

childcare_export_gdf.to_file(
    PROCESSED / "socialservices" / "childcare_impacted_facilities.geojson",
    driver="GeoJSON"
)